In [ ]:
!pip install fastcoref==2.1.1
!pip install spacy

In [15]:
from fastcoref import FCoref
import spacy
from dataclasses import dataclass
from typing import List, Tuple

In [ ]:
model = FCoref(device='cpu')
nlp = spacy.load("en_core_web_sm")

07/11/2026 14:35:00 - INFO - 	 missing_keys: []
07/11/2026 14:35:00 - INFO - 	 unexpected_keys: []
07/11/2026 14:35:00 - INFO - 	 mismatched_keys: []
07/11/2026 14:35:00 - INFO - 	 error_msgs: []
07/11/2026 14:35:00 - INFO - 	 Model Parameters: 90.5M, Transformer: 82.1M, Coref head: 8.4M


In [16]:
@dataclass
class Entity:
    """Класс для хранения упоминаний сущностей"""
    ind: int  # индекс сущности внутри текста
    text: str  # упоминание из текста, сами слова
    words_ind: Tuple[int, ...]  # индексы (номера с 0) слов, соответсвующих упоминанию

In [19]:
def extract_entities(text: str, model=None, nlp=None) -> List[Entity]:
    """
    Извлекает все сущности из текста с разрешением кореференции.
    Args:
        text: входной текст
        model: модель FCoref (если None, создается новая)
        nlp: модель spaCy (если None, создается новая)
    
    Returns:
        List[Entity]: список сущностей с полями ind, text, words_ind
    """
    if model is None:
        model = FCoref(device='cpu')
    if nlp is None:
        nlp = spacy.load("en_core_web_sm")
    
    preds = model.predict(texts=[text])
    clusters = preds[0].get_clusters(as_strings=True)
    
    doc = nlp(text)
    
    def get_word_indices(mention_text: str) -> Tuple[int, ...]:
        indices = []
        mention_lower = mention_text.lower()
        for token in doc:
            if not token.is_punct and not token.is_space:
                token_lower = token.text.lower()
                if token_lower in mention_lower or mention_lower in token_lower:
                    indices.append(token.i)
                elif any(word in mention_lower for word in token_lower.split()):
                    indices.append(token.i)
        return tuple(sorted(set(indices)))
    
    # Сначала найдем все кластеры, испольузем Fcoref, тут только кластеры из 2 и более элементов, добавим в рещультат
    result = []
    all_indices_in_clusters = set()
    
    for cluster_id, cluster in enumerate(clusters):
        for mention in cluster:
            indices = get_word_indices(mention)
            result.append({
                'cluster_id': cluster_id,
                'text': mention,
                'indices': indices
            })
            all_indices_in_clusters.update(indices)
    
    # Теперь найдем все упоминания сущностей, сюда попадают сущности, объединенные в кластер и одиночные сущности
    ner_entities = []
    for ent in doc.ents:
        indices = get_word_indices(ent.text)
        ner_entities.append({
            'text': ent.text,
            'indices': indices
        })
    
    # Удаляем кластеры, оставляем только одиночные сущности
    remaining_ner = []
    for entity in ner_entities:
        entity_indices = set(entity['indices'])
        if not entity_indices.issubset(all_indices_in_clusters):
            remaining_ner.append(entity)
    
    # Добавим одиночные представления в результат
    next_cluster_id = len(set([item['cluster_id'] for item in result]))
    
    for entity in remaining_ner:
        result.append({
            'cluster_id': next_cluster_id,
            'text': entity['text'],
            'indices': entity['indices']
        })
        next_cluster_id += 1
    
    # Поправим нумерацию сущностей
    unique_cluster_ids = sorted(set([item['cluster_id'] for item in result]))
    cluster_mapping = {old_id: new_id for new_id, old_id in enumerate(unique_cluster_ids)}
    
    entities = []
    for item in result:
        entities.append(Entity(
            ind=cluster_mapping[item['cluster_id']],
            text=item['text'],
            words_ind=item['indices']
        ))
    
    return entities

In [22]:
# ПРИМЕРЧИК
model = FCoref(device='cpu')
nlp = spacy.load("en_core_web_sm")

text = """The Anne Frank House has revealed that Anne Frank and her older sister, Margot, likely died at least a month earlier than previously believed. The sisters, who were imprisoned in Nazi concentration camps during the Holocaust, were thought to have died in March 1945, just two weeks before the Bergen-Belsen camp was liberated. However, new research examining archives from the Red Cross, the International Tracing Service, the Bergen-Belsen Memorial, and survivor testimonies suggests that the sisters did not survive until March. The exact dates of their deaths remain unclear, but it is thought that both had symptoms of typhus, the disease they succumbed to, before February 7, 1945."""

entities = extract_entities(text, model=model, nlp=nlp)

for entity in entities:
    print(f"  Entity(ind={entity.ind}, text='{entity.text}', words_ind={entity.words_ind})")

print(f"   Всего упоминаний: {len(entities)}")
print(f"   Всего сущностей: {len(set([e.ind for e in entities]))}")

07/11/2026 15:10:11 - INFO - 	 missing_keys: []
07/11/2026 15:10:11 - INFO - 	 unexpected_keys: []
07/11/2026 15:10:11 - INFO - 	 mismatched_keys: []
07/11/2026 15:10:11 - INFO - 	 error_msgs: []
07/11/2026 15:10:11 - INFO - 	 Model Parameters: 90.5M, Transformer: 82.1M, Coref head: 8.4M
07/11/2026 15:10:11 - INFO - 	 Tokenize 1 inputs...


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

07/11/2026 15:10:12 - INFO - 	 ***** Running Inference on 1 texts *****


Inference:   0%|          | 0/1 [00:00<?, ?it/s]

  Entity(ind=0, text='Anne Frank', words_ind=(1, 2, 7, 8, 20))
  Entity(ind=0, text='her', words_ind=(10,))
  Entity(ind=1, text='Anne Frank and her older sister, Margot', words_ind=(1, 2, 7, 8, 9, 10, 11, 12, 14, 20, 84, 108))
  Entity(ind=1, text='The sisters, who were imprisoned in Nazi concentration camps during the Holocaust', words_ind=(0, 12, 18, 20, 27, 28, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 41, 46, 54, 58, 69, 73, 78, 89, 90, 97, 108, 117))
  Entity(ind=1, text='the sisters', words_ind=(0, 12, 27, 28, 38, 54, 69, 73, 78, 89, 90, 97, 108, 117))
  Entity(ind=1, text='their', words_ind=(0, 27, 38, 54, 69, 73, 78, 89, 97, 101, 117))
  Entity(ind=1, text='both', words_ind=(111,))
  Entity(ind=1, text='they', words_ind=(0, 27, 38, 54, 69, 73, 78, 89, 97, 117, 119))
  Entity(ind=2, text='March 1945, just two weeks before the Bergen-Belsen camp was liberated', words_ind=(0, 18, 20, 27, 38, 47, 48, 50, 51, 52, 53, 54, 55, 57, 58, 59, 60, 69, 73, 78, 79, 81, 89, 95, 97, 117, 123, 1